# 🎯 Ćwiczenie 10: MLlib — Wykrywanie fraudów transakcji kartą płatniczą

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 10
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐⭐⭐ (Zaawansowany)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Tuningować hiperparametry modelu MLlib przy pomocy `ParamGridBuilder` i `CrossValidator`
2. Wybrać właściwą metrykę ewaluacji dla mocno niezbalansowanych danych (AUC-PR vs AUC-ROC)
3. Zastosować ważenie klas (`weightCol`) jako sposób radzenia sobie z class imbalance w Spark
4. Wytrenować i ocenić `GBTClassifier` na dużym zbiorze danych
5. Wyciągnąć najlepsze hiperparametry z `CrossValidatorModel` i zapisać/wczytać `PipelineModel`
6. Policzyć macierz konfuzji przy niestandardowym progu prawdopodobieństwa

## 📋 Wymagania wstępne
- Znajomość PySpark MLlib: `Pipeline`, `VectorAssembler`, `StringIndexer` (Wykład 9)
- Znajomość `BinaryClassificationEvaluator` i podstawowych modeli klasyfikacji

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz, przeczytaj komórki z podpowiedziami (💡 Hint) lub zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

import random
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

random.seed(42)
np.random.seed(42)

spark = (
    SparkSession.builder
    .appName("CardShield-Fraud-Detection")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("✅ Setup complete! SparkSession gotowa 🚀")
print(f"Wersja Sparka: {spark.version}")

## 📖 Kontekst: CardShield 💳🕵️

Pracujesz jako data scientist w firmie **CardShield**, dostawcy systemów antyfraudowych dla banków. Masz ~500 000 transakcji kartą płatniczą z ostatniego miesiąca — ale tylko **1-2%** z nich to fraudy. To klasyczny problem **class imbalance** w skali Big Data: model, który zawsze przewiduje "nie fraud", osiągnie >98% accuracy i będzie kompletnie bezużyteczny!

### Kluczowe koncepty na dziś

| Koncept | Opis | Przykład |
|---------|------|----------|
| `ParamGridBuilder` | buduje siatkę kombinacji hiperparametrów do przetestowania | `ParamGridBuilder().addGrid(rf.numTrees, [50, 100]).build()` |
| `CrossValidator` | trenuje i ocenia model dla każdej kombinacji z siatki, k-fold CV | `CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3)` |
| `TrainValidationSplit` | jak `CrossValidator`, ale pojedynczy podział zamiast k-fold — szybszy, mniej solidny | `TrainValidationSplit(..., trainRatio=0.8)` |
| `weightCol` | kolumna wag przykładów — sposób na class imbalance bez fizycznego undersamplingu | `RandomForestClassifier(weightCol="class_weight")` |
| `areaUnderPR` | AUC dla krzywej Precision-Recall — lepsza metryka niż AUC-ROC przy silnym imbalance | `BinaryClassificationEvaluator(metricName="areaUnderPR")` |

> 🔑 **Zasada kciuka:** Gdy klasa pozytywna to <5% danych, **AUC-PR** mówi Ci znacznie więcej niż AUC-ROC — bo skupia się na tym, jak dobrze model radzi sobie właśnie z rzadką klasą.

In [ ]:
# @title 📊 Generowanie zbioru danych transakcji (~500 000 wierszy, fraud rate ~1-2%)

N = 500_000

merchant_categories = ["elektronika", "paliwo", "gastronomia", "podroze", "apteka", "odziez", "elektronika_online", "kasyno_online"]
countries = ["PL", "DE", "FR", "US", "GB", "NG", "RU", "BR"]

amount = np.round(np.random.lognormal(mean=3.8, sigma=1.1, size=N), 2)
hour = np.random.randint(0, 24, N)
card_country = np.random.choice(countries, N, p=[0.55, 0.15, 0.08, 0.08, 0.06, 0.03, 0.03, 0.02])

# transaction_country zwykle = card_country, ale czasem różny kraj (sygnał fraudu)
different_country_mask = np.random.rand(N) < 0.08
transaction_country = np.where(
    different_country_mask,
    np.random.choice(countries, N),
    card_country,
)

merchant_category = np.random.choice(merchant_categories, N, p=[0.18, 0.15, 0.20, 0.10, 0.12, 0.13, 0.07, 0.05])
transactions_last_hour = np.random.poisson(lam=1.2, size=N)

# Realistyczne sygnały fraudu: inny kraj, wysoka kwota, kasyno_online, dużo transakcji w ostatniej godzinie, nocna pora
base_prob = 0.006
country_mismatch_bonus = np.where(transaction_country != card_country, 0.06, 0.0)
high_amount_bonus = np.where(amount > 300, 0.03, 0.0)
casino_bonus = np.where(merchant_category == "kasyno_online", 0.05, 0.0)
velocity_bonus = np.where(transactions_last_hour >= 4, 0.08, 0.0)
night_bonus = np.where((hour >= 1) & (hour <= 4), 0.02, 0.0)

fraud_prob = np.clip(
    base_prob + country_mismatch_bonus + high_amount_bonus + casino_bonus + velocity_bonus + night_bonus,
    0.0, 0.97,
)
is_fraud = (np.random.rand(N) < fraud_prob).astype(int)

pdf = pd.DataFrame({
    "transaction_id": np.arange(1, N + 1),
    "amount": amount,
    "hour": hour,
    "transaction_country": transaction_country,
    "card_country": card_country,
    "merchant_category": merchant_category,
    "transactions_last_hour": transactions_last_hour,
    "is_fraud": is_fraud,
})

df = spark.createDataFrame(pdf)
df.printSchema()
print(f"📊 Liczba wierszy: {df.count():,}")
df.groupBy("is_fraud").count().show()
fraud_rate = df.filter("is_fraud = 1").count() / df.count()
print(f"🕵️ Fraud rate: {fraud_rate:.2%}")

## 🎓 Guided Example — Pipeline + CrossValidator z ParamGridBuilder

Zbudujmy razem pełny `Pipeline` MLlib (`StringIndexer` → `VectorAssembler` → `RandomForestClassifier`), a następnie owińmy go w `CrossValidator` z małą siatką hiperparametrów, oceniając modele metryką **areaUnderPR** (bo dane są mocno niezbalansowane!).

**Uwaga:** Ten przykład używa własnego, lokalnego podziału danych (`guided_train` / `guided_test`) tylko do demonstracji. W ćwiczeniach stworzysz **własny** oficjalny podział (`train_df` / `test_df`), którego użyjesz we wszystkich kolejnych zadaniach.

In [ ]:
# @title 🎓 Guided Example — uruchom i przestudiuj (może potrwać kilka minut na CrossValidator)

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

categorical_cols = ["transaction_country", "card_country", "merchant_category"]
indexed_cols = [f"{c}_idx" for c in categorical_cols]
numeric_cols = ["amount", "hour", "transactions_last_hour"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="is_fraud", seed=42)

guided_pipeline = Pipeline(stages=indexers + [assembler, rf])

# Mała siatka — na zajęciach chcemy szybki wynik, nie produkcyjny grid search!
guided_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)

guided_cv = CrossValidator(
    estimator=guided_pipeline,
    estimatorParamMaps=guided_grid,
    evaluator=pr_evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

guided_train, guided_test = df.randomSplit([0.7, 0.3], seed=42)
guided_train = guided_train.cache()          # 👈 CrossValidator wielokrotnie odczytuje train — cache przyspiesza CV

guided_cv_model = guided_cv.fit(guided_train)
guided_predictions = guided_cv_model.bestModel.transform(guided_test)

guided_auc_pr = pr_evaluator.evaluate(guided_predictions)
print(f"✅ Guided example — najlepszy model, AUC-PR: {guided_auc_pr:.4f}")
guided_predictions.select("amount", "transaction_country", "card_country", "is_fraud", "prediction", "probability").show(5, truncate=False)

---

## ✏️ Ćwiczenie 1: Oficjalny podział train/test

**Trudność:** ⭐

Podziel `df` na `train_df` (70%) i `test_df` (30%), używając ziarna losowości `seed=42`. Zcache'uj `train_df` — będzie wielokrotnie wykorzystywany przez `CrossValidator` w kolejnych ćwiczeniach. Te dwie zmienne będą używane we **wszystkich** kolejnych ćwiczeniach.

**Oczekiwany wynik:** `train_df` ma ok. 70% wierszy, `test_df` ok. 30% wierszy, a ich suma równa się `df.count()`.

> 💡 **Hint:** Użyj `df.randomSplit([...], seed=...)`, a potem `.cache()` na `train_df`.

In [ ]:
# ✏️ Ćwiczenie 1: Podział train/test

# TODO: podziel df na train_df (70%) i test_df (30%), seed=42, zcache'uj train_df
train_df = ...  # Zastąp ... swoim kodem
test_df = ...  # Zastąp ... swoim kodem

print(f"train_df: {train_df.count():,} wierszy")
print(f"test_df: {test_df.count():,} wierszy")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    train_count = train_df.count()
    test_count = test_df.count()
    total = df.count()

    assert train_count is not ..., "❌ train_df nie zostało uzupełnione"
    assert abs(train_count / total - 0.7) < 0.02, f"❌ train_df powinno mieć ok. 70% wierszy, ma {train_count / total:.2%}"
    assert abs(test_count / total - 0.3) < 0.02, f"❌ test_df powinno mieć ok. 30% wierszy, ma {test_count / total:.2%}"
    assert train_count + test_count == total, "❌ suma train_df + test_df powinna równać się df.count()"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 2: AUC-ROC vs AUC-PR — dlaczego to ma znaczenie?

**Trudność:** ⭐⭐

Weź `guided_predictions` z guided example. Oblicz **obie** metryki: `areaUnderROC` i `areaUnderPR` przy pomocy `BinaryClassificationEvaluator`. Zapisz je jako `auc_roc` i `auc_pr`. Następnie w zmiennej `wyjasnienie` (string, min. 40 znaków) napisz po polsku, dlaczego przy fraud rate ~1-2% AUC-PR jest bardziej wiarygodną metryką niż AUC-ROC.

**Oczekiwany wynik:** `auc_roc` będzie zauważalnie wyższe niż `auc_pr` — to normalne przy silnym class imbalance!

> 💡 **Hint:** AUC-ROC uwzględnia True Negative Rate, którego przy 98-99% klasy negatywnej łatwo "osiągnąć przypadkiem". AUC-PR skupia się tylko na precision/recall klasy pozytywnej.

In [ ]:
# ✏️ Ćwiczenie 2: Porównanie AUC-ROC vs AUC-PR

# TODO: stwórz evaluator z metricName="areaUnderROC" i policz auc_roc na guided_predictions
auc_roc = ...  # Zastąp ... swoim kodem

# TODO: policz auc_pr (może użyć już istniejącego pr_evaluator z guided example)
auc_pr = ...  # Zastąp ... swoim kodem

# TODO: krótkie wyjaśnienie po polsku (min. 40 znaków)
wyjasnienie = "..."  # Zastąp ... swoim tekstem

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")
print(f"Wyjaśnienie: {wyjasnienie}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    assert auc_roc is not ..., "❌ auc_roc nie zostało uzupełnione"
    assert auc_pr is not ..., "❌ auc_pr nie zostało uzupełnione"
    assert 0.0 <= auc_roc <= 1.0, "❌ auc_roc powinno być w zakresie [0, 1]"
    assert 0.0 <= auc_pr <= 1.0, "❌ auc_pr powinno być w zakresie [0, 1]"
    assert wyjasnienie != "..." and len(wyjasnienie) >= 40, "❌ wyjasnienie powinno mieć min. 40 znaków"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 3: Ważenie klas (`weightCol`)

**Trudność:** ⭐⭐⭐

Dodaj do `train_df` kolumnę `class_weight`: `1.0` dla `is_fraud == 0`, oraz `(liczba nie-fraudów / liczba fraudów)` dla `is_fraud == 1` (klasyczne *inverse class frequency weighting*). Zapisz wynik jako `train_weighted`. Następnie zbuduj `Pipeline` analogiczny do guided example, ale z `RandomForestClassifier(weightCol="class_weight", ...)`, wytrenuj go i policz `auc_pr_weighted` na `test_df`.

**Oczekiwany wynik:** `train_weighted` ma dodatkową kolumnę `class_weight`; `auc_pr_weighted` to liczba w [0, 1].

> 💡 **Hint:** Użyj `F.when(F.col("is_fraud") == 1, waga_fraud).otherwise(1.0)`.

In [ ]:
# ✏️ Ćwiczenie 3: Ważenie klas

from pyspark.sql import functions as F

n_fraud = train_df.filter("is_fraud = 1").count()
n_ok = train_df.filter("is_fraud = 0").count()
fraud_weight = n_ok / n_fraud

# TODO: dodaj kolumnę class_weight do train_df -> train_weighted
train_weighted = ...  # Zastąp ... swoim kodem

weighted_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
weighted_assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")

# TODO: stwórz RandomForestClassifier z weightCol="class_weight", numTrees=100, maxDepth=8
weighted_rf = ...  # Zastąp ... swoim kodem

weighted_pipeline = Pipeline(stages=weighted_indexers + [weighted_assembler, weighted_rf])

# TODO: wytrenuj na train_weighted i policz auc_pr_weighted na test_df
weighted_model = ...  # Zastąp ... swoim kodem
auc_pr_weighted = ...  # Zastąp ... swoim kodem

print(f"Waga klasy fraud: {fraud_weight:.1f}")
print(f"AUC-PR (ważony model): {auc_pr_weighted:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    assert "class_weight" in train_weighted.columns, "❌ train_weighted powinno mieć kolumnę class_weight"
    assert auc_pr_weighted is not ..., "❌ auc_pr_weighted nie zostało uzupełnione"
    assert 0.0 <= auc_pr_weighted <= 1.0, "❌ auc_pr_weighted powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 4: `GBTClassifier` z `CrossValidator`

**Trudność:** ⭐⭐⭐⭐

Zbuduj `Pipeline` z `GBTClassifier` (zamiast `RandomForestClassifier`), owiń go w `CrossValidator` z małą siatką `ParamGridBuilder` (np. `maxIter` w `[20, 50]`, `maxDepth` w `[3, 5]`), `numFolds=3`, oceniając `areaUnderPR`. Wytrenuj na `train_df` i zapisz najlepszy model jako `gbt_cv_model`, a AUC-PR na `test_df` jako `auc_pr_gbt`.

**Oczekiwany wynik:** `gbt_cv_model.bestModel` to `PipelineModel`; `auc_pr_gbt` to liczba w [0, 1].

> ⚠️ **Uwaga:** `GBTClassifier` z `CrossValidator` na 500k wierszy może potrwać kilka minut — to normalne, taki jest koszt tuningu na dużych danych!

> 💡 **Hint:** `GBTClassifier` obsługuje tylko klasyfikację binarną — idealnie pasuje do `is_fraud`.

In [ ]:
# ✏️ Ćwiczenie 4: GBTClassifier + CrossValidator

from pyspark.ml.classification import GBTClassifier

gbt_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
gbt_assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")

# TODO: stwórz GBTClassifier(featuresCol="features", labelCol="is_fraud", seed=42)
gbt = ...  # Zastąp ... swoim kodem

gbt_pipeline = Pipeline(stages=gbt_indexers + [gbt_assembler, gbt])

# TODO: zbuduj małą siatkę ParamGridBuilder: gbt.maxIter w [20, 50], gbt.maxDepth w [3, 5]
gbt_grid = ...  # Zastąp ... swoim kodem

# TODO: stwórz CrossValidator (estimator=gbt_pipeline, estimatorParamMaps=gbt_grid, evaluator=pr_evaluator, numFolds=3, parallelism=2, seed=42)
gbt_cv = ...  # Zastąp ... swoim kodem

# TODO: wytrenuj na train_df -> gbt_cv_model
gbt_cv_model = ...  # Zastąp ... swoim kodem

gbt_predictions = gbt_cv_model.bestModel.transform(test_df)
auc_pr_gbt = pr_evaluator.evaluate(gbt_predictions)
print(f"AUC-PR (GBTClassifier, najlepszy z CV): {auc_pr_gbt:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    from pyspark.ml import PipelineModel
    assert isinstance(gbt_cv_model.bestModel, PipelineModel), "❌ gbt_cv_model.bestModel powinno być PipelineModel"
    assert 0.0 <= auc_pr_gbt <= 1.0, "❌ auc_pr_gbt powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 5: Ekstrakcja najlepszych hiperparametrów z CV

**Trudność:** ⭐⭐⭐

Z `gbt_cv_model` wyciągnij ostatni stage najlepszego modelu (`GBTClassificationModel`) i odczytaj z niego wybrane przez `CrossValidator` wartości `maxIter` i `maxDepth`. Zapisz je jako `best_max_iter` i `best_max_depth`.

**Oczekiwany wynik:** obie wartości pochodzą z siatki, którą zdefiniowałeś/aś w Ćwiczeniu 4.

> 💡 **Hint:** `gbt_cv_model.bestModel.stages[-1]` to wytrenowany model; użyj `.getMaxIter()` i `.getMaxDepth()`.

In [ ]:
# ✏️ Ćwiczenie 5: Najlepsze hiperparametry z CV

# TODO: wyciągnij ostatni stage najlepszego modelu
best_gbt_model = ...  # Zastąp ... swoim kodem

# TODO: odczytaj maxIter i maxDepth z best_gbt_model
best_max_iter = ...  # Zastąp ... swoim kodem
best_max_depth = ...  # Zastąp ... swoim kodem

print(f"Najlepszy maxIter: {best_max_iter}")
print(f"Najlepszy maxDepth: {best_max_depth}")

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    assert best_max_iter in [20, 50], f"❌ best_max_iter powinno pochodzić z siatki [20, 50], jest {best_max_iter}"
    assert best_max_depth in [3, 5], f"❌ best_max_depth powinno pochodzić z siatki [3, 5], jest {best_max_depth}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 6: Zapis i odczyt najlepszego `PipelineModel`

**Trudność:** ⭐⭐

Zapisz `gbt_cv_model.bestModel` do ścieżki `"/tmp/cardshield_best_model"` (użyj `.write().overwrite().save(...)`), a następnie wczytaj go z powrotem jako `loaded_model` przy pomocy `PipelineModel.load(...)`. Zweryfikuj, że predykcje wczytanego modelu na `test_df` są identyczne jak przed zapisem.

**Oczekiwany wynik:** `loaded_model` generuje te same predykcje co `gbt_cv_model.bestModel`.

> 💡 **Hint:** `PipelineModel` (i wszystkie stage'y) implementuje `MLWritable`/`MLReadable`.

In [ ]:
# ✏️ Ćwiczenie 6: Zapis i odczyt modelu

model_path = "/tmp/cardshield_best_model"

# TODO: zapisz gbt_cv_model.bestModel pod model_path (nadpisz jeśli istnieje)
...  # Zastąp ... swoim kodem

# TODO: wczytaj model z powrotem jako loaded_model
loaded_model = ...  # Zastąp ... swoim kodem

original_preds = gbt_cv_model.bestModel.transform(test_df).select("transaction_id", "prediction")
loaded_preds = loaded_model.transform(test_df).select("transaction_id", "prediction")

mismatches = original_preds.join(loaded_preds, "transaction_id", "inner") \
    .filter(original_preds["prediction"] != loaded_preds["prediction"]).count()
print(f"Liczba niezgodnych predykcji: {mismatches}")

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    from pyspark.ml import PipelineModel
    assert isinstance(loaded_model, PipelineModel), "❌ loaded_model powinno być PipelineModel"
    assert mismatches == 0, f"❌ Wczytany model powinien dawać identyczne predykcje, znaleziono {mismatches} różnic"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 7: Macierz konfuzji przy niestandardowym progu

**Trudność:** ⭐⭐⭐⭐

Domyślny próg klasyfikacji w Spark to 0.5 — ale przy fraud rate ~1-2% to zwykle zbyt wysoko! Wyciągnij prawdopodobieństwo klasy `1` z kolumny `probability` (to `Vector`, drugi element) do nowej kolumny `prob_fraud`, a następnie stwórz kolumnę `custom_prediction` = `1`, jeśli `prob_fraud > 0.3`, w przeciwnym razie `0`. Policz macierz konfuzji (TP, FP, TN, FN) jako słownik `confusion` na `test_df` przetransformowanym przez `gbt_cv_model.bestModel`.

**Oczekiwany wynik:** `confusion` to słownik z kluczami `"TP"`, `"FP"`, `"TN"`, `"FN"`, których suma = liczba wierszy `test_df`.

> 💡 **Hint:** Użyj UDF-a lub `vector_to_array` z `pyspark.ml.functions`, by wyciągnąć element wektora `probability`.

In [ ]:
# ✏️ Ćwiczenie 7: Macierz konfuzji przy progu 0.3

from pyspark.ml.functions import vector_to_array

threshold_predictions = gbt_cv_model.bestModel.transform(test_df)

# TODO: dodaj kolumnę prob_fraud = drugi element wektora "probability" (indeks 1)
threshold_predictions = threshold_predictions.withColumn(
    "prob_fraud", ...  # Zastąp ... swoim kodem
)

# TODO: dodaj kolumnę custom_prediction = 1, gdy prob_fraud > 0.3, inaczej 0
threshold_predictions = threshold_predictions.withColumn(
    "custom_prediction", ...  # Zastąp ... swoim kodem
)

tp = threshold_predictions.filter("is_fraud = 1 AND custom_prediction = 1").count()
fp = threshold_predictions.filter("is_fraud = 0 AND custom_prediction = 1").count()
tn = threshold_predictions.filter("is_fraud = 0 AND custom_prediction = 0").count()
fn = threshold_predictions.filter("is_fraud = 1 AND custom_prediction = 0").count()

confusion = {"TP": tp, "FP": fp, "TN": tn, "FN": fn}
print(f"Macierz konfuzji (próg 0.3): {confusion}")

In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    assert set(confusion.keys()) == {"TP", "FP", "TN", "FN"}, "❌ confusion powinno mieć klucze TP, FP, TN, FN"
    assert sum(confusion.values()) == test_df.count(), "❌ suma macierzy konfuzji powinna równać się liczbie wierszy test_df"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## 🏆 Bonus Challenge (opcjonalne)

Bank szacuje, że koszt **niewykrytego fraudu (FN)** to średnio **500 zł** (zwrot pieniędzy klientowi), a koszt **fałszywego alarmu (FP)** to **20 zł** (koszt obsługi + niezadowolenie klienta, którego transakcję wstrzymano). Używając `confusion` z Ćwiczenia 7, policz całkowity koszt (`total_cost = FN * 500 + FP * 20`) dla progu 0.3, a następnie powtórz obliczenia macierzy konfuzji i kosztu dla progu **0.1** i **0.5**. Który próg minimalizuje koszt? Zapisz wniosek w komórce Markdown.

In [ ]:
# 🏆 Bonus: analiza kosztowo-korzyściowa progu decyzyjnego

def confusion_at_threshold(predictions_df, threshold: float) -> dict[str, int]:
    preds = predictions_df.withColumn("prob_fraud", vector_to_array("probability")[1])
    preds = preds.withColumn("custom_prediction", (F.col("prob_fraud") > threshold).cast("int"))
    tp = preds.filter("is_fraud = 1 AND custom_prediction = 1").count()
    fp = preds.filter("is_fraud = 0 AND custom_prediction = 1").count()
    tn = preds.filter("is_fraud = 0 AND custom_prediction = 0").count()
    fn = preds.filter("is_fraud = 1 AND custom_prediction = 0").count()
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn}

base_predictions = gbt_cv_model.bestModel.transform(test_df)

for t in [0.1, 0.3, 0.5]:
    c = confusion_at_threshold(base_predictions, t)
    cost = c["FN"] * 500 + c["FP"] * 20
    print(f"Próg {t}: {c} -> koszt = {cost:,} zł")

---

## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Tuning hiperparametrów przy pomocy `ParamGridBuilder` i `CrossValidator`
- ✅ Wybór odpowiedniej metryki (AUC-PR zamiast AUC-ROC) dla mocno niezbalansowanych danych
- ✅ Ważenie klas (`weightCol`) jako alternatywę dla undersamplingu
- ✅ Trenowanie i ewaluację `GBTClassifier` na dużym zbiorze danych
- ✅ Ekstrakcję najlepszych hiperparametrów z modelu wybranego przez CV
- ✅ Zapis i odczyt `PipelineModel`
- ✅ Analizę macierzy konfuzji przy niestandardowym progu decyzyjnym i prostą analizę kosztowo-korzyściową

### 📚 Dalsze materiały
- [Spark ML Tuning Guide](https://spark.apache.org/docs/latest/ml-tuning.html)
- [pyspark.ml.classification.GBTClassifier](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.GBTClassifier.html)

### 🔮 Co dalej?
Na **Wykładzie 11** przechodzimy do **Cloud Computing i GCP** — BigQuery i Dataproc! ☁️